In [0]:
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

df = spark.table("workspace.enigh.gold_hogares").toPandas()
features = [c for c in df.columns if c not in ("folioviv", "foliohog", "ing_cor", "ingreso_alto")]
X = df[features].astype("float64")
y = df["ingreso_alto"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test), "| Variables:", len(features))

# Baseline
base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline -> F1:", f1_score(y_test, base.predict(X_test), zero_division=0))

# Modelo
mlflow.set_registry_uri("databricks-uc")
modelo = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=42)

with mlflow.start_run(run_name="hgb_ingreso_alto"):
    pesos = compute_sample_weight("balanced", y_train)
    modelo.fit(X_train, y_train, sample_weight=pesos)

    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]
    metricas = {
        "f1": f1_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }
    mlflow.log_params({"modelo": "HistGradientBoostingClassifier", "max_iter": 300,
                       "learning_rate": 0.1, "pesos": "balanced", "umbral_ingreso": 99708.17})
    mlflow.log_metrics(metricas)

    firma = infer_signature(X_train, modelo.predict(X_train))
    mlflow.sklearn.log_model(modelo, artifact_path="model", signature=firma,
                             input_example=X_train.head(3),
                             registered_model_name="workspace.enigh.modelo_ingreso_alto")

print("Métricas en prueba:", {k: round(v, 4) for k, v in metricas.items()})
print("Variables del modelo:", features)

{"ts": "2026-11-20 18:06:43.435", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTYyNzU0ODMxOTE4Nzk4NBABIAEyJDAxYTFmYTIzLTkyYTUtNzBmOC04MTIzLTQ0ZDBlMTdlMWU2ZTokZTcxZGViMTctYWVlZC0zYzkyLWJmNjAtNWJhMzY3NDNmZTFkSgwIrZiC2AYQwOLC3QJQAVgBYAFo0sa015DFow2IAQA=.", "context": {}}
{"ts": "2026-11-20 18:06:43.435", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTYyNzU0ODMxOTE4Nzk4NBABIAEyJDAxYTFmYTIzLTkyYTUtNzBmOC04MTIzLTQ0ZDBlMTdlMWU2ZTokZTcxZGViMTctYWVlZC0zYzkyLWJmNjAtNWJhMzY3NDNmZTFkSgwIrZiC2AYQwOLC3QJQAVgBYAFo0sa015DFow2IAQA=.", "context": {}}
{"ts": "2026-11-20 18:06:43.435", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTYyNzU0ODMxOTE4Nzk4NBABIAEyJDAxYTFmYTIzLTkyYTUtNzBmOC04MTIzLTQ0ZDBlMTdlMWU2ZTokZTcxZGViMTctYWVlZC0zYzkyLWJmNjAtNWJhMzY3

Entrenamiento: 73131 | Prueba: 18283 | Variables: 22
Baseline -> F1: 0.0


2026/11/20 18:06:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-adc59796-504b.cloud.databricks.com/ml/experiments/1627548319187984/models/m-b434de4ba849453d8804d9908ee29dae?o=7474656322593618
Successfully registered model 'workspace.enigh.modelo_ingreso_alto'.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.enigh.modelo_ingreso_alto': https://dbc-adc59796-504b.cloud.databricks.com/explore/data/models/workspace/enigh/modelo_ingreso_alto/version/1?o=7474656322593618


Métricas en prueba: {'f1': 0.7002, 'precision': 0.5888, 'recall': 0.8635, 'roc_auc': 0.9355, 'pr_auc': 0.8093}
Variables del modelo: ['entidad', 'tam_loc', 'est_socio', 'clase_hog', 'sexo_jefe', 'edad_jefe', 'educa_jefe', 'tot_integ', 'ocupados', 'gasto_total_tri', 'n_registros', 'n_claves', 'n_grupos', 'pct_no_monetario', 'pct_g_011', 'pct_g_046', 'pct_g_111', 'pct_g_072', 'pct_g_131', 'pct_g_083', 'pct_g_056', 'pct_g_045']
